# Module 6: Decision — RL
### BWSI SGAI 2026

By the end of this module you'll know:
- What the `ActionRecommender` contract is and why it exists
- How to activate `HeuristicRecommender` and edit its rules
- What `RLRecommender` does and how it uses the obs vector from Module 4
- What `DQN.py` and `DQNAgent.py` are (and that you probably don't need to edit them)

---
*Open this notebook in JupyterLab. Press `Alt+R` to enter slideshow mode.*  
*Students: just scroll through normally and run the exercise cells.*

## How does the game decide what to do?

In Module 5, the CNN gave us four probabilities:

```
probs = [0.02, 0.11, 0.84, 0.03]  # zombie, healthy, injured, corpse
```

Now the game has to **act** on that. It needs to answer:
> *Given what the classifier thinks about this humanoid — and the current state of the ambulance — what should I do?*

That decision belongs to an **`ActionRecommender`**.

---

Two recommenders are included out of the box:

| Recommender | Strategy | Requires training? |
|---|---|---|
| `HeuristicRecommender` | Fixed if/elif rules | No |
| `RLRecommender` | Trained DQN policy | Yes (weights pre-loaded) |

Both implement the same interface. That's the point.

## Where does this live in the code?

```
bwsi-sgai/
├── gameplay/
├── data/
├── game_interfaces/
├── models/                    ← this module
│   ├── ActionRecommender.py  ← now  (abstract base class)
│   ├── HeuristicRecommender.py ← now  (rule-based)
│   ├── RLRecommender.py      ← now  (trained DQN)
│   ├── DQN.py                ← now  (neural network)
│   ├── DQNAgent.py           ← now  (agent wrapper)
│   ├── HumanoidClassifier.py
│   ├── CNNClassifier.py
│   └── OracleClassifier.py
├── ui/
├── training/
└── tests/
```

*Module 5 covered the top half of `models/` — perception. This module covers the bottom half — decision.*

*`bwsi-sgai/models/ActionRecommender.py`*

## ActionRecommender — the Contract

```
bwsi-sgai/
├── models/
│   ├── ActionRecommender.py  ← here
│   ├── HeuristicRecommender.py
│   ├── RLRecommender.py
│   └── ...
```

Open `models/ActionRecommender.py`. It's short — about 85 lines including docs.

**What is an Abstract Base Class (ABC)?**  
An ABC defines a *contract* — a set of methods that every subclass must implement. If a subclass forgets to implement an abstract method, Python raises a `TypeError` at instantiation time rather than silently producing a broken object.

Python docs: [Abstract Base Classes](https://docs.python.org/3/library/abc.html)

*`bwsi-sgai/models/ActionRecommender.py`*

## ActionRecommender — Orient

```python
from abc import ABC, abstractmethod
import numpy as np
from gameplay.enums import ActionState
from gameplay.humanoid import Humanoid
from gameplay.scorekeeper import ScoreKeeper

class ActionRecommender(ABC):
    """Decides what action to TAKE for a given humanoid."""

    @abstractmethod
    def recommend(
        self,
        humanoid: Humanoid,
        probs: np.ndarray,
        scorekeeper: ScoreKeeper,
    ) -> ActionState:
        """Recommend an action for the current humanoid."""
```

One abstract method. Any class that inherits from `ActionRecommender` **must** implement `recommend()` — or Python refuses to let you create an instance of it.

*`bwsi-sgai/models/ActionRecommender.py`*

## ActionRecommender — The Interesting Seam

Why does `recommend()` take `probs` as a parameter instead of running `classify()` itself?

**Two reasons:**

1. **Single forward pass per turn.** The classifier has already run once (in `base_env.py`). Passing `probs` in means the recommender gets the result for free — no second inference call.

2. **Decoupling.** The recommender doesn't care *which* classifier produced `probs`. You can swap `CNNClassifier` → `OracleClassifier` and the recommender works identically. You can also test a recommender by passing in hand-crafted `probs` without needing any image at all.

```python
# In base_env.py — the caller:
probs = classifier.classify(humanoid)       # one inference call
action = recommender.recommend(humanoid, probs, scorekeeper)  # uses that result
```

The recommender is downstream of the classifier. It consumes probs; it never produces them.

*`bwsi-sgai/models/ActionRecommender.py`*

## ActionRecommender — Touch

Run the cell below to read the source directly. Notice: you cannot instantiate `ActionRecommender()` — Python will refuse because `recommend()` is abstract.

In [1]:
import os
import sys
from pathlib import Path
%matplotlib inline

# Ensure the project root is the working directory and is on sys.path,
# so local packages (game_interfaces, gameplay, models) are importable
# and relative paths like DATA_DIR="data" resolve correctly.
# Works whether Jupyter was launched from bwsi-sgai/ or lectures/.
_here = Path().resolve()
_root = _here.parent if _here.name == "lectures" else _here
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

In [2]:
import inspect
from models.ActionRecommender import ActionRecommender

print(inspect.getsource(ActionRecommender))

class ActionRecommender(ABC):
    """Decides what action to TAKE for a given humanoid.

    Implementations:
    RLRecommender (uses a trained DQN policy),
    HeuristicRecommender (fixed classifier-based rules).

    The UI displays the returned action string when a recommender is
    injected. In machine mode the recommended action is also auto-executed.

    The caller computes ``probs`` via a
    :class:`~models.HumanoidClassifier.HumanoidClassifier` and passes them
    in — recommenders do not own a classifier.
    """

    @abstractmethod
    def recommend(
        self,
        humanoid: Humanoid,
        probs: np.ndarray,
        scorekeeper: ScoreKeeper,
    ) -> ActionState:
        """Recommend an action for the current humanoid.

        Args:
            humanoid: Humanoid instance (used for value and metadata).
            probs: ``np.ndarray`` of shape ``(4,)`` — probability distribution
                over :class:`~gameplay.enums.State` classes, as returned by
       

> Try running `ActionRecommender()` directly — you'll get a `TypeError: Can't instantiate abstract class`.  
> That's the contract enforcing itself.

## Now: the simplest decision-maker

```
bwsi-sgai/
├── models/
│   ├── ActionRecommender.py
│   ├── HeuristicRecommender.py  ← now
│   ├── RLRecommender.py
│   ├── DQN.py
│   └── DQNAgent.py
```

*`HeuristicRecommender` is the most accessible file in this module — no weights, no training, just if/elif rules you can read and edit right now.*

*`bwsi-sgai/models/HeuristicRecommender.py`*

## HeuristicRecommender — Locate

Open `models/HeuristicRecommender.py`. It's 55 lines.

```
bwsi-sgai/
├── models/
│   ├── HeuristicRecommender.py  ← here
```

**What it is:** a rule-based recommender. No neural network, no weight file, no training step. The decision logic is plain Python `if/elif` — you can read the entire policy in under 10 lines.

**Why it exists:** it's the easiest way to experiment with decision-making. You edit a rule, run the game, and immediately see what changes. No training loop required.

*`bwsi-sgai/models/HeuristicRecommender.py`*

## HeuristicRecommender — Orient

```python
class HeuristicRecommender(ActionRecommender):
    """Rule-based recommender: classifier probs drive fixed action rules."""

    def recommend(
        self,
        humanoid: Humanoid,
        probs: np.ndarray,
        scorekeeper: ScoreKeeper,
    ) -> ActionState:

        if scorekeeper.remaining_time <= 0 or scorekeeper.at_capacity():
            return ActionState.SCRAM

        predicted_idx = int(np.argmax(probs))
        predicted_state = list(State)[predicted_idx]

        if predicted_state in (State.ZOMBIE, State.CORPSE):
            return ActionState.SQUISH
        return ActionState.SAVE
```

The logic in plain English:
- If ambulance is full or time is up → **Scram**
- If the classifier thinks it's a zombie or corpse → **Squish**
- Otherwise → **Save**

Notice: **Skip is never returned.** This recommender is aggressive — it never passes.

*`bwsi-sgai/models/HeuristicRecommender.py`*

## HeuristicRecommender — The Gotcha

`HeuristicRecommender` is **not wired up to any game mode by default.**

All CLI modes (`auto`, `machine`, `human_rl`, `human_cnn_rl`) use `RLRecommender` out of the box.  
To use `HeuristicRecommender`, you have to edit `main.py`.

Here's the change for the **`auto` mode** path (in `_run_auto()`):

```python
# Add this import at the top of main.py:
from models.HeuristicRecommender import HeuristicRecommender

# In _run_auto(), replace:
recommender = RLRecommender()

# With:
recommender = HeuristicRecommender()
```

To activate it in GUI modes, make the same swap inside `_make_components()` for the relevant mode branch.

> This is intentional — `RLRecommender` is the production default. `HeuristicRecommender` is a research target.

*`bwsi-sgai/models/HeuristicRecommender.py`*

## HeuristicRecommender — Touch

Run the cell below to read the `recommend()` source directly.

In [3]:
import inspect
from models.HeuristicRecommender import HeuristicRecommender

hr = HeuristicRecommender()
print(inspect.getsource(hr.recommend))

    def recommend(
        self,
        humanoid: Humanoid,
        probs: np.ndarray,
        scorekeeper: ScoreKeeper,
    ) -> ActionState:
        """Recommend an action using fixed classifier-to-action rules.

        Args:
            humanoid: Humanoid instance (unused by this recommender, but
                part of the :class:`~models.ActionRecommender.ActionRecommender`
                contract).
            probs: 4-float probability vector from the classifier, as
                returned by
                :meth:`~models.HumanoidClassifier.HumanoidClassifier.classify`.
            scorekeeper: ScoreKeeper tracking game time, capacity, and score.

        Returns:
            An :class:`~gameplay.enums.ActionState` member.
        """
        if scorekeeper.remaining_time <= 0 or scorekeeper.at_capacity():
            return ActionState.SCRAM

        predicted_idx = int(np.argmax(probs))
        predicted_state = list(State)[predicted_idx]

        if predicted_state in (S

*`bwsi-sgai/gameplay/enums.py`*

## Valid Return Values for recommend()

`recommend()` must return an `ActionState` member. Run the cell below to see the full list.

In [4]:
from gameplay.enums import ActionState

print("Valid ActionState values:")
for action in ActionState:
    print(f"  ActionState.{action.name} → '{action.value}'")

Valid ActionState values:
  ActionState.SAVE → 'save'
  ActionState.SQUISH → 'squish'
  ActionState.SKIP → 'skip'
  ActionState.SCRAM → 'scram'


> Your edited `recommend()` must return one of these four values.  
> Returning a string (`'save'`) instead of `ActionState.SAVE` will cause a type error downstream.

INSTRUCTOR NOTES:
Ask students: what rule would you write to be *more conservative* — e.g. skip injured humanoids unless the ambulance is less than half full?
Point out that `scorekeeper.at_capacity()` and `scorekeeper.remaining_time` are the two state variables available without any ML.
Also note: `humanoid` is passed in but the current implementation ignores it entirely. Students can use `humanoid.value` if they want value-based rules.

## Now: the trained agent

```
bwsi-sgai/
├── models/
│   ├── ActionRecommender.py
│   ├── HeuristicRecommender.py
│   ├── RLRecommender.py       ← now
│   ├── DQN.py
│   └── DQNAgent.py
```

*`RLRecommender` wraps a trained DQN. It implements the same `recommend()` contract — but instead of if/elif rules, it runs a neural network.*

*`bwsi-sgai/models/RLRecommender.py`*

## RLRecommender — Orient

```python
class RLRecommender(ActionRecommender):
    """Recommends actions using the trained DQN policy."""

    def __init__(self, rl_model_file: str = RL_MODEL_PATH) -> None:
        if not os.path.exists(rl_model_file):
            raise FileNotFoundError(...)
        self.agent = DQNAgent(obs_size=OBS_SIZE, num_actions=4)
        self.agent.load(rl_model_file)

    def recommend(
        self, humanoid, probs, scorekeeper
    ) -> ActionState:
        obs = self._build_obs(humanoid, probs, scorekeeper)
        available = scorekeeper.available_action_space()
        action_idx = self.agent.select_action(obs, greedy=True, available=available)
        return list(ActionState)[action_idx]

    # ... other methods ...
```

**Constructor:** loads weights from `models/weights/baselineRL.pth` — fails fast if the file is missing.  
**`recommend()`:** builds the 15-slot obs vector (covered in Module 4), asks the DQN for a greedy action, maps the integer back to `ActionState`.

*`bwsi-sgai/models/RLRecommender.py`*

## RLRecommender — The obs vector connection

The RL agent doesn't see raw images. It sees a **15-slot observation vector** built from:
- Current game time (binned)
- Ambulance contents (4 slots)
- Classifier's probs for this humanoid (4 slots)
- Which actions are currently available (4 slots)
- Humanoid value (binned)

This is `OBS_SIZE = 15` from `gameplay/constants.py` — the same number you saw in Module 4.

The `probs` passed into `recommend()` slot directly into positions 5–8 of that vector.  
That's why the single-forward-pass design matters: the classifier runs once, and the result goes to both the display **and** the obs vector.

*`bwsi-sgai/models/RLRecommender.py`*

## RLRecommender — Touch

Run the cell below to check whether the baseline RL weights exist on your machine.

In [5]:
import os
from gameplay.constants import RL_MODEL_PATH, OBS_SIZE

weights_exist = os.path.exists(RL_MODEL_PATH)
print(f"RL weights path:   {RL_MODEL_PATH}")
print(f"File exists:       {weights_exist}")
print(f"OBS_SIZE:          {OBS_SIZE}  (input dimension for the DQN)")

if not weights_exist:
    print("\nWeights missing — run: python main.py train --type rl")

RL weights path:   models\weights\baselineRL.pth
File exists:       True
OBS_SIZE:          15  (input dimension for the DQN)


> `RLRecommender` fails at construction time if weights are missing — not silently at inference time.  
> If you trained a new RL agent (Module 7), you can point to the new weight file: `RLRecommender(rl_model_file='path/to/new.pth')`.

## Now: what's inside the trained agent

```
bwsi-sgai/
├── models/
│   ├── ActionRecommender.py
│   ├── HeuristicRecommender.py
│   ├── RLRecommender.py
│   ├── DQN.py        ← now  (the neural network itself)
│   └── DQNAgent.py   ← now  (the agent that wraps it)
```

*You probably won't edit these files. But knowing they exist — and what they do — matters when you're debugging or designing an extension.*

*`bwsi-sgai/models/DQN.py` · `bwsi-sgai/models/DQNAgent.py`*

## DQN.py — The Neural Network

```python
class DQN(nn.Module):
    """A Q-network that maps observations to action-value estimates (Q-values)."""

    def __init__(self, obs_size: int, num_actions: int, hidden_size: int = 128):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(obs_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, num_actions),
            # No softmax — raw Q-values, not probabilities
        )
```

**Input:** the 15-slot obs vector (`obs_size = OBS_SIZE = 15`)  
**Output:** 4 Q-values — one per action (Save, Skip, Squish, Scram)  
**Action selection:** `argmax` of the Q-values → the agent picks the action with the highest expected reward

Unlike the CNN (which outputs probabilities via softmax), the DQN outputs raw scores. Higher score = agent prefers that action.

*`bwsi-sgai/models/DQNAgent.py`*

## DQNAgent.py — The Agent Wrapper

`DQNAgent` wraps the `DQN` network with everything needed to train and run it:

```
DQNAgent
├── policy_net          — the DQN being trained
├── target_net          — a periodic hard copy (stabilizes training)
├── replay_buffer       — stores past (obs, action, reward, next_obs) transitions
├── select_action(obs, greedy, available)
│   └── epsilon-greedy: random during training, argmax at inference
├── train_step()        — sample a mini-batch, compute loss, update policy_net
├── end_episode()       — decay epsilon, copy policy → target every N episodes
├── save(path)          — write policy_net weights to disk
└── load(path)          — load weights from disk (called by RLRecommender)
```

**At inference time** (`RLRecommender.recommend()`): only `select_action(greedy=True)` runs — no training, no epsilon, just argmax.

**At training time** (`training/train_rl.py`): the full loop runs — `train_step()`, `end_episode()`, epsilon decay, target net updates.

*`bwsi-sgai/models/DQN.py` · `bwsi-sgai/models/DQNAgent.py`*

## DQN + DQNAgent — Touch

Run the cell below to instantiate a fresh DQN and DQNAgent and inspect their shapes.

In [6]:
from gameplay.constants import OBS_SIZE
from models.DQN import DQN
from models.DQNAgent import DQNAgent

# The network: 15 inputs → 128 hidden → 128 hidden → 4 outputs
net = DQN(obs_size=OBS_SIZE, num_actions=4)
print("DQN architecture:")
print(net.network)

print()

# The agent (inference-only: no training, just greedy action selection)
agent = DQNAgent(obs_size=OBS_SIZE, num_actions=4)
print(f"DQNAgent ready — epsilon: {agent.epsilon:.2f}  (1.0 = fully random, 0.05 = fully greedy)")
print(f"OBS_SIZE = {OBS_SIZE}  →  {OBS_SIZE} input features → 4 Q-values")

DQN architecture:
Sequential(
  (0): Linear(in_features=15, out_features=128, bias=True)
  (1): ReLU()
  (2): Linear(in_features=128, out_features=128, bias=True)
  (3): ReLU()
  (4): Linear(in_features=128, out_features=4, bias=True)
)

DQNAgent ready — epsilon: 1.00  (1.0 = fully random, 0.05 = fully greedy)
OBS_SIZE = 15  →  15 input features → 4 Q-values


> The DQN here has random weights — it hasn't been trained yet. `RLRecommender` loads the pre-trained weights via `agent.load(path)` in its constructor.

INSTRUCTOR NOTES:
Students who ask "how does it learn" — point them to Module 7 (Training). This module covers inference only.
The key insight here is the shape: 15 inputs → 4 outputs. Those 15 inputs are the obs vector from Module 4.
epsilon=1.0 here because this is a fresh, untrained agent. The loaded baseline will run with greedy=True, bypassing epsilon entirely.

## Exercise: Edit a HeuristicRecommender Rule

**Time: under 5 minutes**

You're going to change one decision rule in `HeuristicRecommender` and compare the outcome against the default.

**Overview:**
1. Edit `main.py` to activate `HeuristicRecommender`
2. Read the current rules in `HeuristicRecommender.py`
3. Change one rule
4. Run an `auto` game and record the result
5. Change the rule back and compare
6. Revert `main.py` to `RLRecommender`

## Exercise — Step 0: Activate HeuristicRecommender

Open `main.py` and make two edits:

**1. Add the import** (near the top, with the other model imports):
```python
from models.HeuristicRecommender import HeuristicRecommender
```

**2. In `_run_auto()`, find this line:**
```python
recommender = RLRecommender()
```
**And replace it with:**
```python
recommender = HeuristicRecommender()
```

Save the file. Now `python main.py -m auto` will use your rules instead of the trained DQN.

## Exercise — Step 1: Read the current rules

Run the cell below to print the current `recommend()` source.

In [7]:
import inspect
from models.HeuristicRecommender import HeuristicRecommender
from gameplay.enums import ActionState

print("Current rules in HeuristicRecommender.recommend():")
print(inspect.getsource(HeuristicRecommender.recommend))

print("\nAvailable actions:", [a.name for a in ActionState])

Current rules in HeuristicRecommender.recommend():
    def recommend(
        self,
        humanoid: Humanoid,
        probs: np.ndarray,
        scorekeeper: ScoreKeeper,
    ) -> ActionState:
        """Recommend an action using fixed classifier-to-action rules.

        Args:
            humanoid: Humanoid instance (unused by this recommender, but
                part of the :class:`~models.ActionRecommender.ActionRecommender`
                contract).
            probs: 4-float probability vector from the classifier, as
                returned by
                :meth:`~models.HumanoidClassifier.HumanoidClassifier.classify`.
            scorekeeper: ScoreKeeper tracking game time, capacity, and score.

        Returns:
            An :class:`~gameplay.enums.ActionState` member.
        """
        if scorekeeper.remaining_time <= 0 or scorekeeper.at_capacity():
            return ActionState.SCRAM

        predicted_idx = int(np.argmax(probs))
        predicted_state = list(Stat

## Exercise — Step 2: Change a rule

Open `models/HeuristicRecommender.py` and modify `recommend()`. Pick one:

**Option A — add a Skip case:**  
Instead of always saving non-zombies, skip low-value humanoids:
```python
if predicted_state in (State.ZOMBIE, State.CORPSE):
    return ActionState.SQUISH
if humanoid.value < 0.4:          # ← add this
    return ActionState.SKIP       # ← and this
return ActionState.SAVE
```

**Option B — always squish, never save:**  
Remove the save branch entirely:
```python
# Replace the last line with:
return ActionState.SQUISH
```

**Option C — your own rule.** Use `scorekeeper.remaining_time`, `scorekeeper.at_capacity()`, `humanoid.value`, or `predicted_state`. Any rule is valid as long as `recommend()` returns an `ActionState`.

Save the file after editing.

## Exercise — Step 3: Run and observe

Open a terminal in the repo root and run:

```bash
python main.py -m auto --log --seed 42
```

Note the output:
- `killed`, `saved`, `saved_value`
- The final score

Then run the cell below to read the action counts from `log.csv`.

In [8]:
import pandas as pd

try:
    log = pd.read_csv("log.csv")
    # Show the last game's action distribution
    last_game = log[log["run_id"] == log["run_id"].iloc[-1]]
    action_counts = last_game["action"].value_counts()
    print("Action counts (last game):")
    print(action_counts.to_string())
    print(f"\nTotal turns: {len(last_game)}")
except FileNotFoundError:
    print("log.csv not found — run: python main.py -m auto --log --seed 42")

Action counts (last game):
action
scram    6

Total turns: 6


## Exercise — Step 4: Compare

Change the rule back to the original and run again:

```bash
python main.py -m auto --log --seed 42
```

Then run the log cell again. Compare:
- Did `save` count go up or down?
- Did `skip` or `squish` appear where they didn't before?
- Did the final score improve?

The default `HeuristicRecommender` has no `skip` at all — so if you added Option A, you should see `skip` appear in the action counts for the first time.

## Exercise — Step 5: Revert main.py

When you're done experimenting, revert `_run_auto()` in `main.py` back to the original:

```python
# In _run_auto(), restore:
recommender = RLRecommender()
```

You can leave the `HeuristicRecommender` import at the top — it doesn't hurt anything.

---

> **For your research project:** `HeuristicRecommender` is your lowest-friction entry point for testing decision hypotheses. You can encode any policy you can describe in plain English — no training loop required. Use it to establish a *rule-based baseline* before comparing against the trained RL agent.

INSTRUCTOR NOTES:
Walk students through Step 0 first — editing main.py — before they touch HeuristicRecommender.py.
Some students will try to instantiate HeuristicRecommender directly from the notebook and call recommend() — this is fine for testing but remind them that game state (scorekeeper) matters.
Common mistake: returning a string ('save') instead of ActionState.SAVE. Show the error message so they know what to look for.
If log.csv doesn't exist: run python main.py -m auto --log --seed 42 in the terminal first.

## Pop Quiz — Navigation

These are navigation questions, not recall. Where do you go?

---

**Q1.** You want the agent to always skip injured humanoids unless the ambulance is empty. Which file and class do you edit — and what else do you have to change first?

A) `models/RLRecommender.py` — edit the `recommend()` method  
B) `models/HeuristicRecommender.py` — edit `recommend()` and also edit `main.py` to activate it  
C) `gameplay/constants.py` — add a new weight  
D) `models/ActionRecommender.py` — edit the abstract method

**Answer: B — `models/HeuristicRecommender.py`**  
Edit `recommend()` there. But that alone won't run — you also need to edit `main.py` to replace `RLRecommender()` with `HeuristicRecommender()` in `_run_auto()` (and/or `_make_components()`).  
Don't edit `ActionRecommender.py` — that's the shared contract, not an implementation.

## Pop Quiz — Navigation

**Q2.** You trained a new RL agent and saved the weights to `models/weights/myAgent.pth`. What's the minimum change to use it in an `auto` game?

A) Edit `models/DQN.py` to point to the new file  
B) Edit `gameplay/constants.py` to update `RL_MODEL_PATH`  
C) Edit `models/DQNAgent.py` — change the default weight path  
D) Run `python main.py -m auto` — it auto-discovers new weights

**Answer: B — `gameplay/constants.py`**  
Update `RL_MODEL_PATH = 'models/weights/myAgent.pth'`. `RLRecommender` reads from `RL_MODEL_PATH` in its constructor — change it there and every mode that uses `RLRecommender` picks up the new weights automatically.  
Alternatively: pass the path explicitly as `RLRecommender(rl_model_file='models/weights/myAgent.pth')` in `main.py` — no constants change needed.

## Pop Quiz — Navigation

**Q3.** Why does `recommend()` take `probs` as a parameter instead of running the CNN itself?

A) The recommender doesn't have access to the image files  
B) Single forward pass per turn; decoupling — recommender works regardless of which classifier produced the probs  
C) The ABC requires it — abstract methods must match the parent signature  
D) It's a performance optimization for GPU inference

**Answer: B — single forward pass + decoupling**  
The classifier runs once in `base_env.py` and the result (`probs`) is passed into `recommend()`. This means: (1) no redundant inference, and (2) the recommender works with any classifier — CNN, Oracle, or one you write yourself.  
You can also pass hand-crafted `probs` in tests without needing any image or model at all.

## What You Now Know

- `models/ActionRecommender.py` — the ABC contract; one abstract method: `recommend(humanoid, probs, scorekeeper) → ActionState`
- `models/HeuristicRecommender.py` — rule-based; **edit here to change decision logic**; must also edit `main.py` to activate it
- `models/RLRecommender.py` — trained DQN policy; loads `baselineRL.pth`; plugs into the same contract
- `models/DQN.py` — 15→128→128→4 feedforward network; outputs Q-values, not probabilities
- `models/DQNAgent.py` — wraps DQN with epsilon-greedy, replay buffer, training loop

The key design fact:
> `recommend()` takes `probs` as a parameter — never calls `classify()` itself. Single forward pass. Full decoupling.

---

**Next: Module 7 — Training**  
Where do the RL weights come from? How do you retrain the DQN from scratch, change the reward structure, and compare reward curves?